# Modelado Relacional — Telco Customer Churn

**Proyecto:** Predicción de Churn en empresa de Telecomunicaciones  
**Fuente:** `database/telco.db` → tabla `telco_clean`  
**Destino:** Modelo Star Schema + tabla puente N:M en el mismo `telco.db`  
**Objetivo:** Implementar 8 tablas normalizadas (3FN) para análisis de hipótesis H1-H10

---
## Sección 0 — Configuración

Importación de librerías, definición de rutas, conexión a SQLite y verificación de la tabla fuente.

In [1]:
import os
import sqlite3
import pandas as pd

# ── Rutas ─────────────────────────────────────────────────────────────────
PATH_DB    = '../database/telco.db'
PATH_DDL   = '../sql/02_ddl.sql'
PATH_CARGA = '../sql/03_carga.sql'

# ── Crear directorio sql/ si no existe ────────────────────────────────────
os.makedirs('../sql', exist_ok=True)

# ── Conectar a telco.db ────────────────────────────────────────────────────
conn = sqlite3.connect(PATH_DB)
conn.execute('PRAGMA foreign_keys = ON')

# ── Verificar que telco_clean existe y tiene 7043 filas ────────────────────
n = pd.read_sql('SELECT COUNT(*) AS n FROM telco_clean', conn).iloc[0, 0]
print(f'Registros en telco_clean: {n:,}')
assert int(n) == 7043, f'ERROR: se esperaban 7043 filas en telco_clean, se encontraron {n}'

Registros en telco_clean: 7,043


---
## Sección 1 — DDL: Creación de Tablas

Definir y ejecutar el DDL completo del modelo relacional. Guardar en `sql/02_ddl.sql`.

**Modelo:** Star Schema con tabla puente N:M para servicios. 8 tablas en total.

In [2]:
# ── Definir DDL completo ───────────────────────────────────────────────────
ddl_sql = '''
-- ─────────────────────────────────────────────
-- DDL — Telco Customer Churn
-- Modelo: Star Schema + tabla puente N:M
-- Normalizacion: 3FN
-- ─────────────────────────────────────────────

-- Eliminar en orden inverso a las dependencias FK
DROP TABLE IF EXISTS bridge_cliente_servicio;
DROP TABLE IF EXISTS fact_cliente;
DROP TABLE IF EXISTS dim_cliente;
DROP TABLE IF EXISTS dim_genero;
DROP TABLE IF EXISTS dim_tipo_contrato;
DROP TABLE IF EXISTS dim_metodo_pago;
DROP TABLE IF EXISTS dim_tipo_internet;
DROP TABLE IF EXISTS dim_catalogo_servicios;

CREATE TABLE dim_genero (
    genero_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    descripcion TEXT    NOT NULL UNIQUE
);

CREATE TABLE dim_cliente (
    cliente_id    TEXT    PRIMARY KEY,
    genero_id     INTEGER NOT NULL,
    SeniorCitizen INTEGER NOT NULL CHECK (SeniorCitizen IN (0,1)),
    Partner       TEXT    NOT NULL CHECK (Partner IN ('Yes','No')),
    Dependents    TEXT    NOT NULL CHECK (Dependents IN ('Yes','No')),
    FOREIGN KEY (genero_id) REFERENCES dim_genero(genero_id)
);

CREATE TABLE dim_tipo_contrato (
    tipo_contrato_id INTEGER PRIMARY KEY AUTOINCREMENT,
    descripcion      TEXT    NOT NULL UNIQUE
);

CREATE TABLE dim_metodo_pago (
    metodo_pago_id INTEGER PRIMARY KEY AUTOINCREMENT,
    descripcion    TEXT    NOT NULL UNIQUE
);

CREATE TABLE dim_tipo_internet (
    tipo_internet_id INTEGER PRIMARY KEY AUTOINCREMENT,
    descripcion      TEXT    NOT NULL UNIQUE
);

CREATE TABLE dim_catalogo_servicios (
    servicio_id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre      TEXT    NOT NULL UNIQUE,
    categoria   TEXT    NOT NULL CHECK (categoria IN ('Telefonia','Internet'))
);

CREATE TABLE bridge_cliente_servicio (
    cliente_id  TEXT    NOT NULL,
    servicio_id INTEGER NOT NULL,
    estado      TEXT    NOT NULL,
    PRIMARY KEY (cliente_id, servicio_id),
    FOREIGN KEY (cliente_id)  REFERENCES dim_cliente(cliente_id),
    FOREIGN KEY (servicio_id) REFERENCES dim_catalogo_servicios(servicio_id)
);

CREATE TABLE fact_cliente (
    cliente_id       TEXT    PRIMARY KEY,
    tipo_contrato_id INTEGER NOT NULL,
    metodo_pago_id   INTEGER NOT NULL,
    tipo_internet_id INTEGER NOT NULL,
    tenure           INTEGER NOT NULL,
    PaperlessBilling TEXT    NOT NULL,
    MonthlyCharges   REAL    NOT NULL,
    TotalCharges     REAL    NOT NULL,
    Churn            TEXT    NOT NULL,
    Churn_num        INTEGER NOT NULL,
    tenure_group     TEXT,
    perfil_familiar  TEXT,
    n_servicios      INTEGER,
    rango_cargo      TEXT,
    segmento_riesgo  INTEGER,
    FOREIGN KEY (cliente_id)       REFERENCES dim_cliente(cliente_id),
    FOREIGN KEY (tipo_contrato_id) REFERENCES dim_tipo_contrato(tipo_contrato_id),
    FOREIGN KEY (metodo_pago_id)   REFERENCES dim_metodo_pago(metodo_pago_id),
    FOREIGN KEY (tipo_internet_id) REFERENCES dim_tipo_internet(tipo_internet_id)
);
'''

print('DDL del modelo relacional (Star Schema 3FN):')
print(ddl_sql)

DDL del modelo relacional (Star Schema 3FN):

-- ─────────────────────────────────────────────
-- DDL — Telco Customer Churn
-- Modelo: Star Schema + tabla puente N:M
-- Normalizacion: 3FN
-- ─────────────────────────────────────────────

-- Eliminar en orden inverso a las dependencias FK
DROP TABLE IF EXISTS bridge_cliente_servicio;
DROP TABLE IF EXISTS fact_cliente;
DROP TABLE IF EXISTS dim_cliente;
DROP TABLE IF EXISTS dim_genero;
DROP TABLE IF EXISTS dim_tipo_contrato;
DROP TABLE IF EXISTS dim_metodo_pago;
DROP TABLE IF EXISTS dim_tipo_internet;
DROP TABLE IF EXISTS dim_catalogo_servicios;

CREATE TABLE dim_genero (
    genero_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    descripcion TEXT    NOT NULL UNIQUE
);

CREATE TABLE dim_cliente (
    cliente_id    TEXT    PRIMARY KEY,
    genero_id     INTEGER NOT NULL,
    SeniorCitizen INTEGER NOT NULL CHECK (SeniorCitizen IN (0,1)),
    Partner       TEXT    NOT NULL CHECK (Partner IN ('Yes','No')),
    Dependents    TEXT    NOT NULL CHECK

In [3]:
# ── Ejecutar DDL ───────────────────────────────────────────────────────────
conn.executescript(ddl_sql)
conn.execute('PRAGMA foreign_keys = ON')

# ── Guardar en sql/02_ddl.sql ─────────────────────────────────────────────
with open(PATH_DDL, 'w', encoding='utf-8') as f:
    f.write(ddl_sql.strip())
print(f'Guardado en {PATH_DDL}')

# ── Verificar las 8 tablas creadas ────────────────────────────────────────
resultado = pd.read_sql(
    '''SELECT name AS tabla FROM sqlite_master WHERE type='table' ORDER BY name''',
    conn
)
print(f'Tablas en telco.db ({len(resultado)} total):')
display(resultado)

tablas_modelo = ['bridge_cliente_servicio','dim_catalogo_servicios','dim_cliente',
                 'dim_genero','dim_metodo_pago','dim_tipo_contrato',
                 'dim_tipo_internet','fact_cliente']
for t in tablas_modelo:
    assert t in resultado['tabla'].values, f'ERROR: tabla {t} no fue creada'

Guardado en ../sql/02_ddl.sql
Tablas en telco.db (16 total):


,tabla
0,bridge_cliente_servicio
1,dim_catalogo_servicios
2,dim_cliente
3,dim_genero
4,dim_metodo_pago
5,dim_tipo_contrato
6,dim_tipo_internet
7,fact_cliente
8,sqlite_sequence
9,telco_clean


**Conclusión Sección 1:** Las 8 tablas del modelo relacional fueron creadas exitosamente en `telco.db`. El DDL fue guardado en `sql/02_ddl.sql`. El modelo implementa Star Schema con `fact_cliente` como tabla de hechos, 5 dimensiones y `bridge_cliente_servicio` para resolver la relación N:M cliente-servicio.

---
## Sección 2 — Carga de Tablas de Catálogo

Insertar los valores únicos de cada dimensión de catálogo desde `telco_clean`. Guardar todos los INSERT en `sql/03_carga.sql`.

In [4]:
# ── Definir todos los INSERT SQL del pipeline de carga ────────────────────

sql_genero = '''
-- 1. dim_genero: valores unicos de gender
INSERT INTO dim_genero (descripcion)
SELECT DISTINCT gender FROM telco_clean ORDER BY gender;
'''

sql_contrato = '''
-- 2. dim_tipo_contrato: valores unicos de Contract
INSERT INTO dim_tipo_contrato (descripcion)
SELECT DISTINCT Contract FROM telco_clean ORDER BY Contract;
'''

sql_pago = '''
-- 3. dim_metodo_pago: valores unicos de PaymentMethod
INSERT INTO dim_metodo_pago (descripcion)
SELECT DISTINCT PaymentMethod FROM telco_clean ORDER BY PaymentMethod;
'''

sql_internet = '''
-- 4. dim_tipo_internet: valores unicos de InternetService
INSERT INTO dim_tipo_internet (descripcion)
SELECT DISTINCT InternetService FROM telco_clean ORDER BY InternetService;
'''

sql_servicios = '''
-- 5. dim_catalogo_servicios: catalogo manual de 8 servicios
INSERT INTO dim_catalogo_servicios (nombre, categoria) VALUES
  ('PhoneService',    'Telefonia'),
  ('MultipleLines',   'Telefonia'),
  ('OnlineSecurity',  'Internet'),
  ('OnlineBackup',    'Internet'),
  ('DeviceProtection','Internet'),
  ('TechSupport',     'Internet'),
  ('StreamingTV',     'Internet'),
  ('StreamingMovies', 'Internet');
'''

# Nota: SeniorCitizen en telco_clean es TEXT ('Senior'/'No Senior') por el ETL.
# Se convierte a INTEGER (1/0) con CASE para cumplir el CHECK constraint de dim_cliente.
sql_cliente = '''
-- 6. dim_cliente: datos demograficos de cada cliente
INSERT INTO dim_cliente (cliente_id, genero_id, SeniorCitizen, Partner, Dependents)
SELECT
  t.customerID,
  g.genero_id,
  CASE WHEN t.SeniorCitizen = 'Senior' THEN 1 ELSE 0 END AS SeniorCitizen,
  t.Partner,
  t.Dependents
FROM telco_clean t
JOIN dim_genero g ON t.gender = g.descripcion;
'''

sql_fact = '''
-- 7. fact_cliente: metricas de facturacion y churn
INSERT INTO fact_cliente (
  cliente_id, tipo_contrato_id, metodo_pago_id, tipo_internet_id,
  tenure, PaperlessBilling, MonthlyCharges, TotalCharges,
  Churn, Churn_num, tenure_group, perfil_familiar,
  n_servicios, rango_cargo, segmento_riesgo
)
SELECT
  t.customerID,
  c.tipo_contrato_id,
  m.metodo_pago_id,
  i.tipo_internet_id,
  t.tenure,
  t.PaperlessBilling,
  t.MonthlyCharges,
  t.TotalCharges,
  t.Churn,
  t.Churn_num,
  t.tenure_group,
  t.perfil_familiar,
  t.n_servicios,
  t.rango_cargo,
  t.segmento_riesgo
FROM telco_clean t
JOIN dim_tipo_contrato c ON t.Contract        = c.descripcion
JOIN dim_metodo_pago   m ON t.PaymentMethod   = m.descripcion
JOIN dim_tipo_internet i ON t.InternetService = i.descripcion;
'''

sql_bridge = '''
-- 8. bridge_cliente_servicio: 7043 clientes x 8 servicios = 56344 filas
INSERT INTO bridge_cliente_servicio (cliente_id, servicio_id, estado)
SELECT t.customerID, s.servicio_id, t.PhoneService
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'PhoneService'
UNION ALL
SELECT t.customerID, s.servicio_id, t.MultipleLines
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'MultipleLines'
UNION ALL
SELECT t.customerID, s.servicio_id, t.OnlineSecurity
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'OnlineSecurity'
UNION ALL
SELECT t.customerID, s.servicio_id, t.OnlineBackup
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'OnlineBackup'
UNION ALL
SELECT t.customerID, s.servicio_id, t.DeviceProtection
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'DeviceProtection'
UNION ALL
SELECT t.customerID, s.servicio_id, t.TechSupport
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'TechSupport'
UNION ALL
SELECT t.customerID, s.servicio_id, t.StreamingTV
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'StreamingTV'
UNION ALL
SELECT t.customerID, s.servicio_id, t.StreamingMovies
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'StreamingMovies';
'''

# ── Guardar carga completa en sql/03_carga.sql ────────────────────────────
encabezado = ('-- Carga completa --- Telco Customer Churn\n'
              '-- Pipeline: catalogos, dimensiones y hechos\n'
              '-- Orden de carga respeta dependencias FK\n\n')
carga_completa = (encabezado + sql_genero + sql_contrato + sql_pago +
                  sql_internet + sql_servicios + sql_cliente +
                  sql_fact + sql_bridge)
with open(PATH_CARGA, 'w', encoding='utf-8') as f:
    f.write(carga_completa)
print(f'Guardado en {PATH_CARGA} ({len(carga_completa):,} caracteres)')
print('Variables SQL definidas: sql_genero, sql_contrato, sql_pago, sql_internet,')
print('                         sql_servicios, sql_cliente, sql_fact, sql_bridge')

Guardado en ../sql/03_carga.sql (3,591 caracteres)
Variables SQL definidas: sql_genero, sql_contrato, sql_pago, sql_internet,
                         sql_servicios, sql_cliente, sql_fact, sql_bridge


In [5]:
# ── INSERT dim_genero ──────────────────────────────────────────────────────
print('SQL a ejecutar:')
print(sql_genero)
conn.executescript(sql_genero)
conn.execute('PRAGMA foreign_keys = ON')
print('Resultado:')
pd.read_sql('SELECT * FROM dim_genero ORDER BY genero_id', conn)

SQL a ejecutar:

-- 1. dim_genero: valores unicos de gender
INSERT INTO dim_genero (descripcion)
SELECT DISTINCT gender FROM telco_clean ORDER BY gender;

Resultado:


,genero_id,descripcion
0,1,Female
1,2,Male


In [6]:
# ── INSERT dim_tipo_contrato ───────────────────────────────────────────────
print('SQL a ejecutar:')
print(sql_contrato)
conn.executescript(sql_contrato)
conn.execute('PRAGMA foreign_keys = ON')
print('Resultado:')
pd.read_sql('SELECT * FROM dim_tipo_contrato ORDER BY tipo_contrato_id', conn)

SQL a ejecutar:

-- 2. dim_tipo_contrato: valores unicos de Contract
INSERT INTO dim_tipo_contrato (descripcion)
SELECT DISTINCT Contract FROM telco_clean ORDER BY Contract;

Resultado:


,tipo_contrato_id,descripcion
0,1,Month-to-month
1,2,One year
2,3,Two year


In [7]:
# ── INSERT dim_metodo_pago ─────────────────────────────────────────────────
print('SQL a ejecutar:')
print(sql_pago)
conn.executescript(sql_pago)
conn.execute('PRAGMA foreign_keys = ON')
print('Resultado:')
pd.read_sql('SELECT * FROM dim_metodo_pago ORDER BY metodo_pago_id', conn)

SQL a ejecutar:

-- 3. dim_metodo_pago: valores unicos de PaymentMethod
INSERT INTO dim_metodo_pago (descripcion)
SELECT DISTINCT PaymentMethod FROM telco_clean ORDER BY PaymentMethod;

Resultado:


,metodo_pago_id,descripcion
0,1,Bank transfer (automatic)
1,2,Credit card (automatic)
2,3,Electronic check
3,4,Mailed check


In [8]:
# ── INSERT dim_tipo_internet ───────────────────────────────────────────────
print('SQL a ejecutar:')
print(sql_internet)
conn.executescript(sql_internet)
conn.execute('PRAGMA foreign_keys = ON')
print('Resultado:')
pd.read_sql('SELECT * FROM dim_tipo_internet ORDER BY tipo_internet_id', conn)

SQL a ejecutar:

-- 4. dim_tipo_internet: valores unicos de InternetService
INSERT INTO dim_tipo_internet (descripcion)
SELECT DISTINCT InternetService FROM telco_clean ORDER BY InternetService;

Resultado:


,tipo_internet_id,descripcion
0,1,DSL
1,2,Fiber optic
2,3,No


In [9]:
# ── INSERT dim_catalogo_servicios ──────────────────────────────────────────
print('SQL a ejecutar:')
print(sql_servicios)
conn.executescript(sql_servicios)
conn.execute('PRAGMA foreign_keys = ON')
print('Resultado:')
pd.read_sql('SELECT * FROM dim_catalogo_servicios ORDER BY servicio_id', conn)

SQL a ejecutar:

-- 5. dim_catalogo_servicios: catalogo manual de 8 servicios
INSERT INTO dim_catalogo_servicios (nombre, categoria) VALUES
  ('PhoneService',    'Telefonia'),
  ('MultipleLines',   'Telefonia'),
  ('OnlineSecurity',  'Internet'),
  ('OnlineBackup',    'Internet'),
  ('DeviceProtection','Internet'),
  ('TechSupport',     'Internet'),
  ('StreamingTV',     'Internet'),
  ('StreamingMovies', 'Internet');

Resultado:


,servicio_id,nombre,categoria
0,1,PhoneService,Telefonia
1,2,MultipleLines,Telefonia
2,3,OnlineSecurity,Internet
3,4,OnlineBackup,Internet
4,5,DeviceProtection,Internet
5,6,TechSupport,Internet
6,7,StreamingTV,Internet
7,8,StreamingMovies,Internet


**Conclusión Sección 2:** Las 5 tablas de catálogo cargadas desde `telco_clean`:

| Tabla | Registros |
|---|---|
| `dim_genero` | 2 |
| `dim_tipo_contrato` | 3 |
| `dim_metodo_pago` | 4 |
| `dim_tipo_internet` | 3 |
| `dim_catalogo_servicios` | 8 |

Archivo `sql/03_carga.sql` guardado con el pipeline completo de carga.

---
## Sección 3 — Carga de dim_cliente

Insertar los datos demográficos de los 7,043 clientes con JOIN a `dim_genero`.

> **Nota técnica:** `SeniorCitizen` en `telco_clean` es TEXT (`'Senior'`/`'No Senior'`) por conversión del ETL (Paso 1). Se transforma a INTEGER (1/0) con `CASE` para cumplir el `CHECK (SeniorCitizen IN (0,1))` del DDL.

In [10]:
# ── Mostrar SQL de inserción de dim_cliente ────────────────────────────────
print('SQL a ejecutar:')
print(sql_cliente)

SQL a ejecutar:

-- 6. dim_cliente: datos demograficos de cada cliente
INSERT INTO dim_cliente (cliente_id, genero_id, SeniorCitizen, Partner, Dependents)
SELECT
  t.customerID,
  g.genero_id,
  CASE WHEN t.SeniorCitizen = 'Senior' THEN 1 ELSE 0 END AS SeniorCitizen,
  t.Partner,
  t.Dependents
FROM telco_clean t
JOIN dim_genero g ON t.gender = g.descripcion;



In [11]:
# ── Ejecutar INSERT ────────────────────────────────────────────────────────
conn.executescript(sql_cliente)
conn.execute('PRAGMA foreign_keys = ON')

# ── Verificar conteo ───────────────────────────────────────────────────────
n_clientes = pd.read_sql('SELECT COUNT(*) AS total FROM dim_cliente', conn).iloc[0, 0]
print(f'Registros en dim_cliente: {n_clientes:,}')
assert int(n_clientes) == 7043, f'ERROR: se esperaban 7043, se insertaron {n_clientes}'

# ── Muestra con JOIN a dim_genero ─────────────────────────────────────────
print('\nPrimeros 5 registros (con nombre de género):')
pd.read_sql('''
SELECT d.cliente_id, g.descripcion AS genero,
       d.SeniorCitizen, d.Partner, d.Dependents
FROM dim_cliente d
JOIN dim_genero g ON d.genero_id = g.genero_id
LIMIT 5
''', conn)

Registros en dim_cliente: 7,043

Primeros 5 registros (con nombre de género):


,cliente_id,genero,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No


**Conclusión Sección 3:** 7,043 registros cargados en `dim_cliente`. La conversión de `SeniorCitizen` TEXT → INTEGER satisface el CHECK constraint. El JOIN con `dim_genero` resuelve la FK correctamente sin registros huérfanos.

---
## Sección 4 — Carga de fact_cliente

Insertar las métricas de facturación y churn para los 7,043 clientes con JOIN a las 3 dimensiones de FK. Validar KPIs contra el EDA y el ETL.

In [12]:
# ── Mostrar SQL de inserción de fact_cliente ───────────────────────────────
print('SQL a ejecutar:')
print(sql_fact)

SQL a ejecutar:

-- 7. fact_cliente: metricas de facturacion y churn
INSERT INTO fact_cliente (
  cliente_id, tipo_contrato_id, metodo_pago_id, tipo_internet_id,
  tenure, PaperlessBilling, MonthlyCharges, TotalCharges,
  Churn, Churn_num, tenure_group, perfil_familiar,
  n_servicios, rango_cargo, segmento_riesgo
)
SELECT
  t.customerID,
  c.tipo_contrato_id,
  m.metodo_pago_id,
  i.tipo_internet_id,
  t.tenure,
  t.PaperlessBilling,
  t.MonthlyCharges,
  t.TotalCharges,
  t.Churn,
  t.Churn_num,
  t.tenure_group,
  t.perfil_familiar,
  t.n_servicios,
  t.rango_cargo,
  t.segmento_riesgo
FROM telco_clean t
JOIN dim_tipo_contrato c ON t.Contract        = c.descripcion
JOIN dim_metodo_pago   m ON t.PaymentMethod   = m.descripcion
JOIN dim_tipo_internet i ON t.InternetService = i.descripcion;



In [13]:
# ── Ejecutar INSERT ────────────────────────────────────────────────────────
conn.executescript(sql_fact)
conn.execute('PRAGMA foreign_keys = ON')

# ── Verificar conteo ───────────────────────────────────────────────────────
n_fact = pd.read_sql('SELECT COUNT(*) AS total FROM fact_cliente', conn).iloc[0, 0]
print(f'Registros en fact_cliente: {n_fact:,}')
assert int(n_fact) == 7043, f'ERROR: se esperaban 7043, hay {n_fact}'

# ── Validar KPIs contra valores del EDA ───────────────────────────────────
df_kpis = pd.read_sql('''
SELECT
  COUNT(*)                                                              AS total_clientes,
  SUM(Churn_num)                                                        AS total_cancelados,
  ROUND(AVG(Churn_num)*100, 2)                                          AS tasa_cancelacion_pct,
  ROUND(SUM(MonthlyCharges), 2)                                         AS revenue_mensual_total,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS revenue_en_riesgo,
  ROUND(AVG(tenure), 1)                                                 AS tenure_promedio,
  ROUND(AVG(MonthlyCharges), 2)                                         AS cargo_promedio
FROM fact_cliente
''', conn)
display(df_kpis)

# ── Assertions ────────────────────────────────────────────────────────────
kpi = df_kpis.iloc[0]
assert int(kpi['total_clientes'])   == 7043,  'ERROR: total_clientes no coincide'
assert int(kpi['total_cancelados']) == 1869,  'ERROR: total_cancelados no coincide'
assert abs(kpi['tasa_cancelacion_pct']   - 26.54)     < 0.01, 'ERROR: tasa_cancelacion no coincide'
assert abs(kpi['revenue_mensual_total']  - 456116.60) < 0.50, 'ERROR: revenue_mensual no coincide'
assert abs(kpi['revenue_en_riesgo']      - 139130.85) < 0.50, 'ERROR: revenue_en_riesgo no coincide'
assert abs(kpi['tenure_promedio']        - 32.4)      < 0.10, 'ERROR: tenure_promedio no coincide'
assert abs(kpi['cargo_promedio']         - 64.76)     < 0.01, 'ERROR: cargo_promedio no coincide'
print('Todos los KPIs coinciden con el EDA y el ETL')

Registros en fact_cliente: 7,043


,total_clientes,total_cancelados,tasa_cancelacion_pct,revenue_mensual_total,revenue_en_riesgo,tenure_promedio,cargo_promedio
0,7043,1869,26.54,456116.6,139130.85,32.4,64.76


Todos los KPIs coinciden con el EDA y el ETL


### Tabla Comparativa — fact_cliente vs EDA

| Métrica | Valor esperado | Valor modelo | ¿Coincide? |
|---|---|---|---|
| Total clientes | 7,043 | 7,043 | ✅ |
| Total cancelados | 1,869 | 1,869 | ✅ |
| Tasa cancelación | 26.54% | 26.54% | ✅ |
| Revenue mensual total | $456,116.60 | $456,116.60 | ✅ |
| Revenue en riesgo | $139,130.85 | $139,130.85 | ✅ |
| Tenure promedio | 32.4 meses | 32.4 meses | ✅ |
| Cargo promedio | $64.76 | $64.76 | ✅ |

**Conclusión Sección 4:** `fact_cliente` cargada con 7,043 registros. Todos los KPIs coinciden exactamente con el EDA y el ETL. Las 3 FKs (tipo_contrato, metodo_pago, tipo_internet) resueltas correctamente.

---
## Sección 5 — Carga de bridge_cliente_servicio

Insertar una fila por cada combinación cliente-servicio: **7,043 clientes × 8 servicios = 56,344 filas**.

In [14]:
# ── Mostrar SQL de inserción de bridge_cliente_servicio ────────────────────
print('SQL a ejecutar (UNION ALL de 8 servicios):')
print(sql_bridge)

SQL a ejecutar (UNION ALL de 8 servicios):

-- 8. bridge_cliente_servicio: 7043 clientes x 8 servicios = 56344 filas
INSERT INTO bridge_cliente_servicio (cliente_id, servicio_id, estado)
SELECT t.customerID, s.servicio_id, t.PhoneService
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'PhoneService'
UNION ALL
SELECT t.customerID, s.servicio_id, t.MultipleLines
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'MultipleLines'
UNION ALL
SELECT t.customerID, s.servicio_id, t.OnlineSecurity
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'OnlineSecurity'
UNION ALL
SELECT t.customerID, s.servicio_id, t.OnlineBackup
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'OnlineBackup'
UNION ALL
SELECT t.customerID, s.servicio_id, t.DeviceProtection
  FROM telco_clean t JOIN dim_catalogo_servicios s ON s.nombre = 'DeviceProtection'
UNION ALL
SELECT t.customerID, s.servicio_id, t.TechSupport
  FROM telco_clean t JOIN dim_catalogo_servi

In [15]:
# ── Ejecutar INSERT ────────────────────────────────────────────────────────
conn.executescript(sql_bridge)
conn.execute('PRAGMA foreign_keys = ON')

# ── Verificar conteo ───────────────────────────────────────────────────────
n_bridge = pd.read_sql('SELECT COUNT(*) AS total FROM bridge_cliente_servicio', conn).iloc[0, 0]
print(f'Registros en bridge_cliente_servicio: {n_bridge:,}')
assert int(n_bridge) == 56344, f'ERROR: se esperaban 56344, hay {n_bridge}'  # 7043 clientes x 8 servicios

# ── Distribución por servicio y estado ────────────────────────────────────
print('\nDistribución por servicio y estado de contratación:')
pd.read_sql('''
SELECT
  s.nombre    AS servicio,
  s.categoria,
  b.estado,
  COUNT(*)    AS cantidad,
  ROUND(COUNT(*) * 100.0 / 7043, 1) AS pct_clientes
FROM bridge_cliente_servicio b
JOIN dim_catalogo_servicios s ON b.servicio_id = s.servicio_id
GROUP BY s.nombre, s.categoria, b.estado
ORDER BY s.nombre, b.estado
''', conn)

Registros en bridge_cliente_servicio: 56,344

Distribución por servicio y estado de contratación:


,servicio,categoria,estado,cantidad,pct_clientes
0,DeviceProtection,Internet,No,3095,43.9
1,DeviceProtection,Internet,No internet service,1526,21.7
2,DeviceProtection,Internet,Yes,2422,34.4
3,MultipleLines,Telefonia,No,3390,48.1
4,MultipleLines,Telefonia,No phone service,682,9.7
5,MultipleLines,Telefonia,Yes,2971,42.2
6,OnlineBackup,Internet,No,3088,43.8
7,OnlineBackup,Internet,No internet service,1526,21.7
8,OnlineBackup,Internet,Yes,2429,34.5
9,OnlineSecurity,Internet,No,3498,49.7


**Conclusión Sección 5:** 56,344 filas cargadas en `bridge_cliente_servicio` (7,043 × 8). La PK compuesta `(cliente_id, servicio_id)` garantiza unicidad. La tabla puente permite consultar el estado de contratación de cualquier servicio sin columnas duplicadas en `fact_cliente`.

---
## Sección 6 — Validación de Integridad Referencial

Verificar que no existen registros huérfanos en ninguna clave foránea. Todas las validaciones deben retornar 0.

In [16]:
# ── Validaciones de integridad referencial ─────────────────────────────────
validaciones = [
    ('fact_cliente -> dim_cliente',
     '''SELECT COUNT(*) AS huerfanos FROM fact_cliente f
        LEFT JOIN dim_cliente d ON f.cliente_id = d.cliente_id
        WHERE d.cliente_id IS NULL'''),
    ('bridge_cliente_servicio -> dim_cliente',
     '''SELECT COUNT(*) AS huerfanos FROM bridge_cliente_servicio b
        LEFT JOIN dim_cliente d ON b.cliente_id = d.cliente_id
        WHERE d.cliente_id IS NULL'''),
    ('fact_cliente -> dim_tipo_contrato',
     '''SELECT COUNT(*) AS huerfanos FROM fact_cliente f
        LEFT JOIN dim_tipo_contrato c ON f.tipo_contrato_id = c.tipo_contrato_id
        WHERE c.tipo_contrato_id IS NULL'''),
    ('fact_cliente -> dim_metodo_pago',
     '''SELECT COUNT(*) AS huerfanos FROM fact_cliente f
        LEFT JOIN dim_metodo_pago m ON f.metodo_pago_id = m.metodo_pago_id
        WHERE m.metodo_pago_id IS NULL'''),
    ('fact_cliente -> dim_tipo_internet',
     '''SELECT COUNT(*) AS huerfanos FROM fact_cliente f
        LEFT JOIN dim_tipo_internet i ON f.tipo_internet_id = i.tipo_internet_id
        WHERE i.tipo_internet_id IS NULL'''),
]

print('Validacion de integridad referencial (todos deben ser 0):\n')
for descripcion, sql in validaciones:
    huerfanos = int(pd.read_sql(sql, conn).iloc[0, 0])
    estado = 'OK   ' if huerfanos == 0 else 'ERROR'
    print(f'  {estado} | {descripcion:50s} | Huerfanos: {huerfanos}')
    assert huerfanos == 0, f'ERROR: {descripcion} tiene {huerfanos} registros huerfanos'

Validacion de integridad referencial (todos deben ser 0):

  OK    | fact_cliente -> dim_cliente                        | Huerfanos: 0
  OK    | bridge_cliente_servicio -> dim_cliente             | Huerfanos: 0
  OK    | fact_cliente -> dim_tipo_contrato                  | Huerfanos: 0
  OK    | fact_cliente -> dim_metodo_pago                    | Huerfanos: 0
  OK    | fact_cliente -> dim_tipo_internet                  | Huerfanos: 0


### Tabla de Validación FK

| Validación | Huérfanos | ¿OK? |
|---|---|---|
| `fact_cliente` → `dim_cliente` | 0 | ✅ |
| `bridge_cliente_servicio` → `dim_cliente` | 0 | ✅ |
| `fact_cliente` → `dim_tipo_contrato` | 0 | ✅ |
| `fact_cliente` → `dim_metodo_pago` | 0 | ✅ |
| `fact_cliente` → `dim_tipo_internet` | 0 | ✅ |

**Conclusión Sección 6:** Integridad referencial verificada al 100%. El modelo está listo para consultas analíticas.

---
## Sección 7 — Resumen del Modelo

Consolidación de tablas creadas, conteos finales y decisiones de diseño.

In [17]:
# ── Conteo final de todas las tablas del modelo ────────────────────────────
tablas_con_orden = [
    'dim_genero', 'dim_tipo_contrato', 'dim_metodo_pago',
    'dim_tipo_internet', 'dim_catalogo_servicios',
    'dim_cliente', 'fact_cliente', 'bridge_cliente_servicio',
]
conteos = {
    t: int(pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0, 0])
    for t in tablas_con_orden
}
df_resumen_modelo = pd.DataFrame(conteos.items(), columns=['tabla', 'registros'])
print(f'Total registros en el modelo: {sum(conteos.values()):,}')
df_resumen_modelo

Total registros en el modelo: 70,450


,tabla,registros
0,dim_genero,2
1,dim_tipo_contrato,3
2,dim_metodo_pago,4
3,dim_tipo_internet,3
4,dim_catalogo_servicios,8
5,dim_cliente,7043
6,fact_cliente,7043
7,bridge_cliente_servicio,56344


### Estadísticas Finales del Modelo

```
Tablas creadas: 8
─────────────────────────────────────────────────────
dim_genero              :         2 registros
dim_tipo_contrato       :         3 registros
dim_metodo_pago         :         4 registros
dim_tipo_internet       :         3 registros
dim_catalogo_servicios  :         8 registros
dim_cliente             :     7,043 registros
fact_cliente            :     7,043 registros
bridge_cliente_servicio :    56,344 registros
─────────────────────────────────────────────────────
Total registros en el modelo: 70,450
```

### Decisiones de Diseño

| Decisión | Alternativa Descartada | Justificación |
|---|---|---|
| Star Schema + tabla puente | Star Schema puro sin N:M | Cumplir 3FN y resolver correctamente la relación N:M cliente-servicio |
| Claves subrogadas en catálogos | Claves naturales (texto) | Estabilidad, performance en JOINs y desacoplamiento del texto fuente |
| SCD Type 1 (snapshot) | SCD Type 2 (historial) | Dataset es snapshot sin historial temporal disponible |
| `dim_genero` separada | `gender` como atributo en `dim_cliente` | Eliminar dependencia transitiva — `gender` como texto en `dim_cliente` viola 3FN |
| `bridge_cliente_servicio` | Dos tablas dim separadas | Resuelve N:M correctamente con PK compuesta — flexible para nuevos servicios |
| Columnas calculadas en `fact_cliente` | Calcular en cada query | Performance analítica — `tenure_group`, `n_servicios`, `rango_cargo` calculados una sola vez |
| `PRAGMA foreign_keys = ON` | Confianza implícita en la aplicación | Garantía de integridad a nivel de base de datos — detecta errores en el pipeline |

### Archivos Generados

| Archivo | Contenido |
|---|---|
| `sql/02_ddl.sql` | DDL completo: 8 DROP + 8 CREATE TABLE |
| `sql/03_carga.sql` | Pipeline de carga: 8 INSERT completos |
| `database/telco.db` | Base de datos con 8 tablas y 70,450 registros |

**Próximo paso:** `sql/04_kpis.sql` y `notebooks/04_analisis.ipynb` — análisis de hipótesis H1-H10 usando el modelo relacional.